# 04 Baseline Poisson Naïve Bayes Decoder (ThresholdCrossings, γ = 0)

This notebook implements the **baseline decoder** described in Issar et al. (2020),
*J Neurophysiol* 123:1472–1485.

## Context
The paper trains a neural network (NAS — Not A Sorter) that assigns each waveform
a spike-likelihood score Pspike ∈ [0,1].  A γ-threshold gates which waveforms are
used for decoding.  **γ = 0 means all threshold crossings are used** — this is the
baseline from which all NAS improvements are measured.

This notebook computes that baseline: a **Poisson Naïve Bayes decoder**
(paper §Materials and Methods, "Decoding") applied to the ThresholdCrossings
stream across all 312 sessions of Monkey N.

## Inputs
- `/kaggle/input/datasets/katakuricharlotte/02-session-index-results/tables_session_index/session_master_index.csv`
- `/kaggle/input/datasets/katakuricharlotte/02-session-index-results/tables_session_index/decoder_trial_table.csv`
- `/kaggle/input/datasets/katakuricharlotte/03-stream-inventory-results/tables_stream_inventory/candidate_stream_manifest.csv`
- Raw NWB files: `/kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N`

## Outputs
- `tables_baseline_decoder/session_baseline_accuracy.csv`
- `tables_baseline_decoder/decoder_unit_table.csv`
- `meta_baseline_decoder/baseline_decoder_report.json`
- `figures_baseline_decoder/fig01_*.{png,pdf}` … `fig07_*`

## Figures (paper-aligned)
| Figure | Paper analogue |
|---|---|
| fig01: Sample session accuracy report | Fig. 4A/B (γ = 0 baseline point) |
| fig02: Session-wise accuracy × trial count | Fig. 4A/B dual-axis |
| fig03: Accuracy vs days since first session | Fig. 5C |
| fig04: Early vs late session distributions | Fig. 5D |
| fig05: Session distribution histograms (4-panel) | Fig. 4C–F |
| fig06: Mean ± 1 SE normalised accuracy | Fig. 5E (γ = 0 slice) |
| fig07: Accuracy vs recording quality proxy | Fig. 5A/B |

In [ ]:
!pip install pynwb h5py

In [ ]:
import os
import re
import json
import math
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

warnings.filterwarnings("ignore")
from pynwb import NWBHDF5IO

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PAPER-STYLE GLOBAL SETTINGS — Issar et al. 2020 (J Neurophysiol)
# ─────────────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "axes.linewidth": 1.2,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "legend.fontsize": 11,
    "legend.frameon": True,
    "legend.edgecolor": "0.4",
    "grid.linestyle": ":",
    "grid.linewidth": 0.7,
    "grid.alpha": 0.85,
})

C_BLACK   = "#1a1a1a"
C_MAROON  = "#8B2222"     # TC / primary accuracy
C_BLUE    = "#2166AC"     # baseline reference dot (paper Fig 4A)
C_GREEN   = "#1B7837"     # maximum / best accuracy
C_GRAY    = "#888888"     # chance level / control
C_ORANGE  = "#E08214"     # SpikingBandPower stream
C_EARLY   = "#2166AC"     # early sessions (blue, as in paper Fig 5E)
C_LATE    = "#8B2222"     # late sessions  (maroon, as in paper Fig 5E)

MARKER_TC  = "o"
MARKER_SBP = "s"
CHANCE     = 0.125        # 1/8 directions

def paper_axes(ax, top=True, right=True):
    ax.minorticks_on()
    ax.grid(True, which="major", linestyle=":", linewidth=0.8, alpha=0.85)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.6)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
    ax.tick_params(which="both", direction="in", top=top, right=right)

np.random.seed(42)

## Paths

In [ ]:
DATASET_DIR = Path("/kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N")

IN_IDX_DIR    = Path("/kaggle/input/datasets/katakuricharlotte/02-session-index-results/tables_session_index")
IN_STREAM_DIR = Path("/kaggle/input/datasets/katakuricharlotte/03-stream-inventory-results/tables_stream_inventory")

SESSION_MASTER_CSV    = IN_IDX_DIR    / "session_master_index.csv"
DECODER_TRIAL_CSV     = IN_IDX_DIR    / "decoder_trial_table.csv"
CANDIDATE_STREAM_CSV  = IN_STREAM_DIR / "candidate_stream_manifest.csv"

OUT_FIG_DIR   = Path("/kaggle/working/figures_baseline_decoder")
OUT_TABLE_DIR = Path("/kaggle/working/tables_baseline_decoder")
OUT_META_DIR  = Path("/kaggle/working/meta_baseline_decoder")

for d in [OUT_FIG_DIR, OUT_TABLE_DIR, OUT_META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SAMPLE_NWB = DATASET_DIR / "sub-Monkey-N_ses-20200127_ecephys.nwb"

print("DATASET_DIR     :", DATASET_DIR)
print("SESSION_MASTER  :", SESSION_MASTER_CSV)
print("DECODER_TRIALS  :", DECODER_TRIAL_CSV)
print("CANDIDATE_STREAM:", CANDIDATE_STREAM_CSV)
print("SAMPLE_NWB      :", SAMPLE_NWB)

In [ ]:
assert DATASET_DIR.exists(),           "Missing dataset directory"
assert SESSION_MASTER_CSV.exists(),    "Missing session master CSV (run notebook 02)"
assert DECODER_TRIAL_CSV.exists(),     "Missing decoder trial table (run notebook 02)"
assert CANDIDATE_STREAM_CSV.exists(),  "Missing candidate stream manifest (run notebook 03)"
assert SAMPLE_NWB.exists(),            "Missing sample NWB"

session_df  = pd.read_csv(SESSION_MASTER_CSV)
trial_df    = pd.read_csv(DECODER_TRIAL_CSV)
stream_df   = pd.read_csv(CANDIDATE_STREAM_CSV)

session_df["session_date"] = pd.to_datetime(session_df["session_date"], errors="coerce")
if "days_since_first_session" not in session_df.columns:
    d0 = session_df["session_date"].min()
    session_df["days_since_first_session"] = (session_df["session_date"] - d0).dt.days

tc_stream_df = stream_df[stream_df["name"] == "ThresholdCrossings"].copy().reset_index(drop=True)

print("session_df shape  :", session_df.shape)
print("trial_df shape    :", trial_df.shape)
print("stream_df shape   :", stream_df.shape)
print("TC manifest rows  :", len(tc_stream_df))
display(session_df.head(3))
display(trial_df.head(3))

## Helper functions

### Neural data extraction

For each session we load the `ThresholdCrossings` `ElectricalSeries` from the NWB file.
The stream shape is **(T × 96)**, where T is the number of samples.  We align samples to
trial timing using the `timestamps` array (if present) or the stream's `rate` and
`starting_time`.

### Decoding window

Following Issar et al. (2020) §Materials and Methods:
> *"Waveforms classified as spikes from 300 ms to 500 ms after fixation
> (i.e., 50 ms after stimulus offset) were used for decoding."*

The memory-guided saccade task timing is:
- Fixation: 200 ms → Stimulus flash: 50 ms → Delay: 500 ms
- Decoding window: [300 ms, 500 ms] relative to fixation onset

For the NWB trial table, `start_time` marks trial start (≈ fixation onset).
We extract TC counts in `[start + 0.30, start + 0.50]` seconds.

### Target label extraction

`mrs_target_position` stores (x, y) target coordinates.  We compute the polar angle
and discretise into **8 octants** (0°, 45°, 90°, …, 315°) for the center-out task.

In [ ]:
def safe_attr(obj, attr, default=None):
    try:
        return getattr(obj, attr)
    except Exception:
        return default

def get_tc_stream_meta(nwb_path):
    """Return (rate_hz, starting_time, has_timestamps) for ThresholdCrossings."""
    with NWBHDF5IO(str(nwb_path), mode="r", load_namespaces=True) as io:
        nwb = io.read()
        for obj in nwb.all_children():
            if safe_attr(obj, "name") == "ThresholdCrossings":
                rate  = safe_attr(obj, "rate",          None)
                t0    = safe_attr(obj, "starting_time", 0.0)
                ts    = safe_attr(obj, "timestamps",    None)
                has_ts = ts is not None
                return rate, t0, has_ts
    return None, 0.0, False

def angle_to_octant(angle_deg):
    """Snap angle to nearest of 8 octants: 0,45,90,...,315."""
    octant = int((angle_deg + 22.5) % 360 // 45)
    return octant  # 0..7

def extract_target_labels(trials_df):
    """
    Discretise mrs_target_position (or index_target_position) into 8 direction labels.
    Returns a Series of integer labels 0–7, or NaN if unavailable.
    """
    # Try index_target_position first (already discrete in CO tasks)
    if "index_target_position" in trials_df.columns:
        labels = pd.to_numeric(trials_df["index_target_position"], errors="coerce")
        if labels.notna().mean() > 0.8 and labels.nunique() <= 12:
            return labels.astype("Int64")

    # Try mrs_target_position as (x, y) string or separate columns
    for col in ["mrs_target_position", "target_position"]:
        if col not in trials_df.columns:
            continue
        try:
            pos = trials_df[col].apply(
                lambda v: [float(x) for x in str(v).strip("[]() ").split()] if pd.notna(v) else [np.nan, np.nan]
            )
            x_vals = pos.apply(lambda p: p[0] if len(p) >= 1 else np.nan)
            y_vals = pos.apply(lambda p: p[1] if len(p) >= 2 else np.nan)
            angles = np.degrees(np.arctan2(y_vals, x_vals)) % 360
            return angles.apply(lambda a: angle_to_octant(a) if np.isfinite(a) else np.nan).astype("Int64")
        except Exception:
            continue

    return pd.Series([np.nan] * len(trials_df), dtype="Float64")

In [ ]:
def load_tc_data(nwb_path, rate_hz=None):
    """
    Load ThresholdCrossings data array and time axis.
    Returns (data: ndarray T×C, times: ndarray T, rate: float).
    """
    with NWBHDF5IO(str(nwb_path), mode="r", load_namespaces=True) as io:
        nwb = io.read()
        for obj in nwb.all_children():
            if safe_attr(obj, "name") != "ThresholdCrossings":
                continue
            data = safe_attr(obj, "data", None)
            if data is None:
                continue
            data = np.array(data, dtype=np.float32)      # T × C
            rate  = safe_attr(obj, "rate",          None)
            t0    = safe_attr(obj, "starting_time", 0.0) or 0.0
            ts    = safe_attr(obj, "timestamps",    None)

            if ts is not None:
                times = np.array(ts, dtype=np.float64)
            elif rate is not None and rate > 0:
                times = t0 + np.arange(data.shape[0]) / float(rate)
            else:
                # Fallback: assume 1 sample = 1 ms
                times = t0 + np.arange(data.shape[0]) * 1e-3

            return data, times, float(rate) if rate else 1000.0

    return None, None, None

In [ ]:
def extract_trial_features(tc_data, times, trials_df, win_start=0.30, win_end=0.50):
    """
    Bin TC counts per trial within the decoding window.

    Parameters
    ----------
    tc_data   : (T, C) float32  — raw ThresholdCrossings array
    times     : (T,) float64    — absolute timestamps (s)
    trials_df : DataFrame       — must have start_time, stop_time columns
    win_start : float           — seconds after trial start (default 0.30 s)
    win_end   : float           — seconds after trial start (default 0.50 s)

    Returns
    -------
    X : (n_trials, C) float32   — spike count features
    valid_mask : bool array     — True where window was found in data
    """
    n_trials  = len(trials_df)
    n_chans   = tc_data.shape[1]
    X         = np.full((n_trials, n_chans), np.nan, dtype=np.float32)
    valid_mask = np.zeros(n_trials, dtype=bool)

    t_start_arr = trials_df["start_time"].values.astype(float)
    t_stop_arr  = trials_df["stop_time"].values.astype(float)
    t_min       = times[0]
    t_max       = times[-1]

    for i, (t0, t_stop) in enumerate(zip(t_start_arr, t_stop_arr)):
        w0 = t0 + win_start
        w1 = t0 + win_end

        # Fallback: use full trial if window exceeds trial duration
        trial_dur = t_stop - t0
        if trial_dur < (win_end - win_start):
            w0 = t0
            w1 = t_stop

        if w0 > t_max or w1 < t_min:
            continue

        idx = np.where((times >= w0) & (times < w1))[0]
        if len(idx) == 0:
            continue

        # Spike count = sum of absolute values (TC can be signed or unsigned)
        window_data = tc_data[idx, :]
        X[i, :]     = np.sum(np.abs(window_data), axis=0)
        valid_mask[i] = True

    return X, valid_mask

In [ ]:
def poisson_nb_decoder(X_train, y_train, X_test, min_rate=0.25, n_classes=8):
    """
    Poisson Naïve Bayes decoder (Issar et al. 2020 §Methods).

    Parameters
    ----------
    X_train, X_test : (n, C) spike counts
    y_train         : (n,) integer class labels 0..n_classes-1
    min_rate        : channels with mean count below this are dropped
    n_classes       : number of target directions

    Returns
    -------
    y_pred : (n_test,) predicted labels
    n_units : number of channels kept
    """
    classes = np.arange(n_classes)
    EPS     = 1e-8

    # Compute mean spike count per class per channel
    lam = np.zeros((n_classes, X_train.shape[1]))
    for c in classes:
        mask = y_train == c
        if mask.sum() == 0:
            lam[c, :] = EPS
        else:
            lam[c, :] = np.maximum(X_train[mask].mean(axis=0), EPS)

    # Drop low-firing units (paper: < 0.25 spikes/s across all trials)
    mean_rate = X_train.mean(axis=0)
    keep      = mean_rate >= min_rate
    if keep.sum() == 0:
        keep = np.ones(X_train.shape[1], dtype=bool)

    lam_k     = lam[:, keep]
    X_test_k  = X_test[:, keep]

    # Poisson log-likelihood: sum_u [ n_u * log(λ_cu) - λ_cu ]
    n_test    = X_test_k.shape[0]
    log_probs = np.zeros((n_test, n_classes))
    for c in classes:
        log_probs[:, c] = (
            X_test_k * np.log(lam_k[c, :] + EPS) - lam_k[c, :]
        ).sum(axis=1)

    y_pred = np.argmax(log_probs, axis=1)
    return y_pred, int(keep.sum())

In [ ]:
def cross_validate_decoder(X, y, n_folds=5, min_rate=0.25, n_classes=8):
    """
    5-fold stratified cross-validation of Poisson NB decoder.

    Returns
    -------
    acc    : float  — mean accuracy across folds
    std    : float  — std of per-fold accuracy
    n_units: int    — median units kept
    """
    n       = len(y)
    idx     = np.arange(n)
    np.random.shuffle(idx)
    folds   = np.array_split(idx, n_folds)

    fold_accs   = []
    fold_units  = []

    for k in range(n_folds):
        test_idx  = folds[k]
        train_idx = np.concatenate([folds[j] for j in range(n_folds) if j != k])

        X_train, y_train = X[train_idx], y[train_idx]
        X_test,  y_test  = X[test_idx],  y[test_idx]

        # Only keep folds with enough class coverage
        classes_present = np.unique(y_train)
        if len(classes_present) < n_classes:
            continue

        y_pred, n_units = poisson_nb_decoder(
            X_train, y_train, X_test,
            min_rate=min_rate, n_classes=n_classes
        )
        fold_accs.append((y_pred == y_test).mean())
        fold_units.append(n_units)

    if len(fold_accs) == 0:
        return np.nan, np.nan, 0

    return float(np.mean(fold_accs)), float(np.std(fold_accs)), int(np.median(fold_units))

## Inspect sample NWB — confirm stream structure before full run

Before decoding all 312 sessions we verify that the ThresholdCrossings stream
is accessible, the timestamps / rate are sensible, and the trial table contains
the expected timing and target columns.

In [ ]:
print("Probing sample NWB:", SAMPLE_NWB.name)

with NWBHDF5IO(str(SAMPLE_NWB), mode="r", load_namespaces=True) as io:
    nwb = io.read()

    # ── ThresholdCrossings ──────────────────────────────────────────────────
    tc_obj = None
    for obj in nwb.all_children():
        if safe_attr(obj, "name") == "ThresholdCrossings":
            tc_obj = obj
            break

    assert tc_obj is not None, "ThresholdCrossings not found in sample NWB"

    tc_shape    = np.array(tc_obj.data).shape
    tc_rate     = safe_attr(tc_obj, "rate",          None)
    tc_t0       = safe_attr(tc_obj, "starting_time", 0.0)
    tc_ts       = safe_attr(tc_obj, "timestamps",    None)
    tc_unit     = safe_attr(tc_obj, "unit",          "unknown")
    tc_has_ts   = tc_ts is not None

    print("\n── ThresholdCrossings ──────────────────")
    print(f"  Shape       : {tc_shape}")
    print(f"  Rate (Hz)   : {tc_rate}")
    print(f"  Starting t  : {tc_t0}")
    print(f"  Has timestamps: {tc_has_ts}")
    print(f"  Unit        : {tc_unit}")
    if tc_has_ts:
        ts_arr = np.array(tc_ts)
        print(f"  Timestamps  : [{ts_arr[0]:.4f} … {ts_arr[-1]:.4f}] s")
        print(f"  Median dt   : {np.median(np.diff(ts_arr))*1000:.3f} ms → rate ≈ {1/np.median(np.diff(ts_arr)):.1f} Hz")

    # ── Trials ──────────────────────────────────────────────────────────────
    assert nwb.trials is not None, "No trials table in sample NWB"
    trials_sample = nwb.trials.to_dataframe()
    print("\n── Trials table ────────────────────────")
    print(f"  Shape  : {trials_sample.shape}")
    print(f"  Columns: {list(trials_sample.columns)}")
    display(trials_sample.head(5))

In [ ]:
# ── Infer effective sampling rate ────────────────────────────────────────────
if tc_has_ts:
    ts_arr   = np.array(tc_ts)
    dt_med   = float(np.median(np.diff(ts_arr)))
    RATE_HZ  = 1.0 / dt_med
else:
    RATE_HZ  = float(tc_rate) if tc_rate else 1000.0

print(f"Effective sampling rate used for decoding: {RATE_HZ:.2f} Hz")

# ── Compute window width in samples ─────────────────────────────────────────
WIN_START = 0.30   # s after trial start
WIN_END   = 0.50   # s after trial start
WIN_SAMPS = int((WIN_END - WIN_START) * RATE_HZ)
print(f"Decoding window: [{WIN_START*1000:.0f}, {WIN_END*1000:.0f}] ms  → {WIN_SAMPS} samples")

In [ ]:
# ── Test target-label extraction on sample ────────────────────────────────
labels_sample = extract_target_labels(trials_sample)
print("Label sample (first 20):", labels_sample.head(20).tolist())
print("Unique labels:", sorted(labels_sample.dropna().unique().tolist()))
print("Label counts:\n", labels_sample.value_counts().sort_index())

In [ ]:
# ── Quick decode test on sample NWB ─────────────────────────────────────────
tc_data, times, rate_detected = load_tc_data(SAMPLE_NWB)
print(f"TC data shape  : {tc_data.shape}")
print(f"Times range    : [{times[0]:.4f}, {times[-1]:.4f}] s")
print(f"Rate detected  : {rate_detected:.2f} Hz")

# Extract features
X_sample, valid_mask = extract_trial_features(tc_data, times, trials_sample,
                                               win_start=WIN_START, win_end=WIN_END)
y_sample_raw = extract_target_labels(trials_sample).values

# Keep only valid trials with a label
keep = valid_mask & np.array([v is not None and not (isinstance(v, float) and np.isnan(v))
                                for v in y_sample_raw])
X_valid = X_sample[keep].astype(np.float32)
y_valid = y_sample_raw[keep].astype(int)

print(f"\nValid trials for decoding : {keep.sum()} / {len(trials_sample)}")
print(f"X_valid shape             : {X_valid.shape}")
print(f"y_valid unique classes    : {sorted(np.unique(y_valid).tolist())}")

acc_sample, std_sample, units_sample = cross_validate_decoder(
    X_valid, y_valid, n_folds=5, n_classes=len(np.unique(y_valid))
)
print(f"\nSample session baseline accuracy : {acc_sample*100:.2f}% ± {std_sample*100:.2f}%")
print(f"Chance level                     : {CHANCE*100:.1f}%")
print(f"Units kept                       : {units_sample}")